In [1]:
import os


os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from functools import partial

import numpy as onp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from msmjax.benchmark_tools import (
    path_input_structures,
    evaluate_structure_with_lammps_p3m,
)
from msmjax.convenience import (
    suggest_msm_params,
    set_up_kernel_fns,
)
from msmjax.shortrange import make_pair_term_fn, make_compute_U0
from msmjax.flexcell import make_flex_cell_U1plus_fn

# Load structure

In [2]:
structures = onp.load(path_input_structures / "structures_1000.npz")

i_structure = 0
pos = structures["positions"][i_structure].astype(onp.float64)
chg = structures["charges"][i_structure].astype(onp.float64)
cell = structures["cells"][i_structure].astype(onp.float64)

box_lengths = onp.diag(cell)
n_structures = len(structures["positions"])
n_particles = pos.shape[0]
avg_particle_spacing = (onp.linalg.det(cell) / n_particles) ** (1 / 3)

# Function definitions

## Energy calculators

In [3]:
calc_energy_ref_nonperiodic = make_pair_term_fn(
    kernel_fn=lambda x: 1.0 / x, pbc=(False, False, False)
)
# The `cell` parameter is unnecessary and without effect in the absence of periodicity
calc_energy_ref_nonperiodic = jax.jit(
    partial(calc_energy_ref_nonperiodic, cell=onp.eye(3))
)


@jax.jit
def calc_forces_ref_nonperiodic(positions, charges):
    return -jax.grad(calc_energy_ref_nonperiodic)(positions, charges)

In [4]:
calc_e_and_f_ref_periodic = partial(
    evaluate_structure_with_lammps_p3m,
    lammps_executable="/home/florian/Downloads/lammps-static/bin/lmp",
    max_neighbors_one_atom=10000,
)

In [5]:
def set_up_flex_cell_msm_energy_fn(reference_cell, pbc, **msm_params):
    kernel_fns = set_up_kernel_fns(
        box_lengths=box_lengths, pbcs=pbc, **msm_params
    )
    calc_U0 = make_compute_U0(kernel_fns=kernel_fns, pbc=pbc)
    calc_U1plus = make_flex_cell_U1plus_fn(
        kernel_fns=kernel_fns,
        pbc=pbc,
        reference_cell=reference_cell,
        **msm_params,
    )

    @jax.jit  # TODO: jit here or later?
    def calc_energy(positions, charges, cell):
        return calc_U0(positions, charges, cell) + calc_U1plus(
            positions, charges, cell
        )

    return calc_energy

## Other helper functions

In [6]:
def calc_rmse(y_pred, y_true):
    return onp.sqrt(((y_pred - y_true) ** 2).mean())


def calc_relative_rmse_percent(y_pred, y_true):
    return calc_rmse(y_pred, y_true) / y_true.std() * 100

# Orthogonal: Isotropic strain

In [7]:
lower = 0.5
upper = 2.0
stepsize = 0.05
n_steps = int(onp.round((1.4 - 0.6) / 0.05))
all_deformations = onp.linspace(lower, upper, n_steps)

In [8]:
energies_ref_nonperiodic = onp.full_like(all_deformations, onp.nan)
forces_ref_nonperiodic = onp.full(
    (len(all_deformations), n_particles, 3), onp.nan
)
energies_ref_periodic = onp.full_like(energies_ref_nonperiodic, onp.nan)
forces_ref_periodic = onp.full_like(forces_ref_nonperiodic, onp.nan)
for i, deformation in enumerate(all_deformations):
    pos_deformed = pos * deformation
    cell_deformed = cell * deformation
    energies_ref_nonperiodic[i] = calc_energy_ref_nonperiodic(
        pos_deformed, chg
    )
    forces_ref_nonperiodic[i] = calc_forces_ref_nonperiodic(pos_deformed, chg)
    try:
        e_ref, f_ref_periodic = calc_e_and_f_ref_periodic(
            pos_deformed, chg, cell_deformed
        )
        energies_ref_periodic[i] = e_ref
        forces_ref_periodic[i] = f_ref_periodic
    except Exception as e:
        print(repr(e))
        continue

results_ref = {
    "pbc-False": {
        "energies": energies_ref_nonperiodic,
        "forces": forces_ref_nonperiodic,
    },
    "pbc-True": {
        "energies": energies_ref_periodic,
        "forces": forces_ref_periodic,
    },
}

ValueError('EOF reached without finding energy')
ValueError('EOF reached without finding energy')
ValueError('EOF reached without finding energy')
ValueError('EOF reached without finding energy')


In [ ]:
R_CUT = 3.5
SPACINGS = [1.0, 0.8, 0.6]

results_msm_flex = {}
for pbc_key, pbc in zip(
    ["pbc-False", "pbc-True"], [(False, False, False), (True, True, True)]
):
    results_msm_flex[pbc_key] = {}
    for spacing in SPACINGS:
        msm_params = suggest_msm_params(
            box_lengths=box_lengths,
            pbcs=pbc,
            n_particles=n_particles,
            level_one_gridspacing=spacing * avg_particle_spacing,
            level_zero_cutoff=R_CUT * avg_particle_spacing,
        )
        calc_energy_msm = set_up_flex_cell_msm_energy_fn(
            reference_cell=cell, pbc=pbc, **msm_params
        )

        @jax.jit
        def calc_forces_msm(positions, charges, cell):
            return -jax.grad(calc_energy_msm)(positions, charges, cell)

        energies_msm_flex = onp.full_like(all_deformations, onp.nan)
        rmses_msm_flex = onp.full_like(all_deformations, onp.nan)
        for i, deformation in enumerate(all_deformations):
            pos_deformed = pos * deformation
            cell_deformed = cell * deformation
            energies_msm_flex[i] = calc_energy_msm(
                pos_deformed, chg, cell_deformed
            )
            f_msm = calc_forces_msm(pos_deformed, chg, cell_deformed)
            rmses_msm_flex[i] = calc_relative_rmse_percent(
                y_pred=f_msm, y_true=results_ref[pbc_key]["forces"][i]
            )

        actual_spacing = msm_params["level_one_gridspacing"]
        results_msm_flex[pbc_key][actual_spacing] = {
            "energies": energies_msm_flex,
            "force_rmses": rmses_msm_flex,
        }

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Highest-level grid not coarser than simulation box size ("criterion 1.")
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")

Suggested value for max grid level L was found according to criterion/criteria:
(if multiple criteria listed, that means they agree on the suggested L)
*) Cutoff at highest-but-one level not too large relative to simulation box size ("criterion 2.")



In [ ]:
figsize = (8, 5)
plot_titles = {
    "pbc-False": "Non-periodic: flexible-cell MSM vs. reference\n(explicit all-pairs summation)",
    "pbc-True": "Periodic: flexible-cell MSM vs. reference\n(LAMMPS with 'pair_style coul/long 10.0', 'kspace_style pppm 1e-5')",
}

In [ ]:
for key_pbcs in ["pbc-False", "pbc-True"]:
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(plot_titles[key_pbcs])
    ax.set_xlabel("stretch ratio (relative to cell used for MSM setup)")
    ax.set_ylabel("energy")
    for h, res in results_msm_flex[key_pbcs].items():
        # TODO: different label if periodic?
        ax.scatter(
            all_deformations,
            res["energies"],
            label=f"flex-cell MSM with $h={h:.2f}$",
            marker="x",
        )
    ax.scatter(
        all_deformations,
        results_ref[key_pbcs]["energies"],
        color="black",
        facecolor="none",
        label="reference",
        zorder=-10,
    )
    ax.axvline(1.0, color="gray")
    ax.legend()
    # fig.savefig(f"energies_{key_pbcs}.pdf")
    plt.show()

In [ ]:
for key_pbcs in ["pbc-False", "pbc-True"]:
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(plot_titles[key_pbcs])
    ax.set_xlabel("stretch ratio (relative to cell used for MSM setup)")
    ax.set_ylabel("force RMSE / stdev of reference forces (in %)")
    for h, res in results_msm_flex[key_pbcs].items():
        ax.scatter(
            all_deformations,
            res["force_rmses"],
            label=f"flex-cell MSM with $h={h:.2f}$",
            marker="x",
        )
    ax.set_ylim([0, ax.get_ylim()[1]])
    ax.axvline(1.0, color="gray")
    ax.legend()
    # fig.savefig(f"force_rsmes_{key_pbcs}.pdf")
    plt.show()

# Orthogonal: Anisotropic strain

# General parallelepipeds